In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os

from astropy.io import fits
from astropy.constants import c
from astropy.table import QTable

import pyimfit
import pandas as pd

import argparse
from pathlib import Path

import astropy.units as u
import subprocess
import traceback as tb
import sys
import json

from parse_results import *
from astropy.visualization import quantity_support

In [ ]:

master_table = QTable.read("/home/ryans/Projects/Photometric Decomp/Analysis/master_table.csv", data_start=2)
master_table["NAME"] = master_table['\ufeffNAME']
master_table = master_table.to_pandas()

In [ ]:
p = Path("/home/ryans/Projects/Photometric Decomp/Outputs/RESULTS_OF_DECOMPOSITION/")
SGAtable = QTable.read(os.path.join("/home/ryans/Projects/Photometric Decomp/Analysis/", "SGA-2020.fits"), format="fits", hdu=1)
TRACTOR = QTable.read(os.path.join("/home/ryans/Projects/Photometric Decomp/Analysis/", "SGA-2020.fits"), format="fits", hdu=2)
# master_table = astropy.io.ascii.read("/home/ryans/Projects/Photometric Decomp/Analysis/master_table.csv")
galmarks = json.load(open(os.path.join(p, "galmarks.json")))

total_bad_fit = 0
total_fit = 0
bound_sticking = 0

galaxies = []
bands = ["g", "r"]

structure = os.walk(p)
for root, dirs, files in structure:
    if len(glob.glob(os.path.join(root,"*.fits"))) != 0: # Check to see if we have entered a folder with galaxy data
        galaxies.append([Path(root).resolve(), os.path.basename(os.path.dirname(root))])

fit_type = "2_sersic"

galaxy_names = [os.path.basename(galaxy[0]) for galaxy in galaxies]
galaxy_name_counts = dict.fromkeys(galaxy_names, 0)
for galaxy_name in galaxy_names:
    galaxy_name_counts[galaxy_name] += 1
for galaxy_name in galaxy_name_counts.keys():
    if galaxy_name_counts[galaxy_name] > 1:
        print(galaxy_name)

all_results =  get_all_results(galaxies, bands, fit_type, galmarks)
total_fit = len(galaxy_name_counts.keys())-1

thresh = 1
total_bad_fit = len(all_results[all_results["Reduced ChiSq"] > thresh].groups.indices)

print(f"Total fit: {total_fit}")
print(f"Total poor fit: {total_bad_fit} ({total_bad_fit/total_fit * 100:.2f}% bad)")
print(f"Total parameter bounds sticking: {bound_sticking}")

In [ ]:
pixscale = 0.262 * u.arcsec

In [ ]:
def nmgyfluxSB_to_magSB(fluxSB):
    zero_point_star_equiv = u.zero_point_flux(3631.1 * u.Jy)
    return (fluxSB*u.arcsec**2).to(u.mag("AB"), zero_point_star_equiv).value * u.mag("AB/arcsec**2")

def nmgymagSB_to_fluxSB(magSB):
    zero_point_star_equiv = u.zero_point_flux(3631.1 * u.Jy)
    return (magSB.value*u.ABmag).to(u.nmgy, zero_point_star_equiv).value * u.nmgy/u.arcsec**2

# x = 22.5 * u.mag("AB/arcsec**2")
# nmgyfluxSB_to_magSB(nmgymagSB_to_fluxSB(x))

## Plotting

In [ ]:
type_colors = {
    "PR": "red",
    "PB": "green",
    "PH": "blue"
}
type_ls = {
    "PR": "-",
    "PB": "--",
    "PH": "-."
}
gal_types = ["PR", "PH", "PB"]
linewidth = 1.5
dpi = 300

### Flux or Luminosity Ratio

In [ ]:
# a = all_results[all_results["Function Label"] == "Host"]
# plt.hist(a["Flux Ratio"])
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

for gal_type in gal_types:
    a = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    bins = np.arange(0, 1 + 0.05, 0.05)
    val, bins = np.histogram(a["Flux Ratio"], bins=bins)
    plt.stairs(val/len(a), edges=bins, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=gal_type, linewidth=linewidth, fill=True, alpha=0.5)

plt.xlabel(r"$r$ Band ${f_{Polar}}/{f_{Host}}$")
plt.ylabel("Fraction of Sample")
plt.legend()
plt.show()

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (0, 1)
# step = 0.05
nbins = 20
bins = np.linspace(r[0], r[1], nbins) * u.ABmag
ep = 0.01

baseline = np.zeros(np.size(bins)-1)
for k, gal_type in enumerate(gal_types):
    a = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]

    vals, bins = np.histogram(a["Flux Ratio"], bins=bins)
    # plt.stairs(vals/ngals+baseline,baseline=baseline, edges=bins, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    plt.stairs(vals/ngals+baseline, edges=bins + ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    # baseline += vals/ngals

plt.xlabel(r"$r$ Band ${f_{Polar}}/{f_{Host}}$")
plt.ylabel("Fraction of Sample")
plt.legend()
plt.show()

### Abs mag

In [ ]:
from astropy.cosmology import Planck18, WMAP9
import numpy as np

cosmo = Planck18

$m-M = 5\log d -5 + A + K_{Corr}$

$m-M_{Corr} = 5\log d -5 - A - K_{Corr}$

$M_{Corr} = M - A - K_{Corr}$

In [ ]:
DLs = []
DAs = []
zs = []
scales = []
K_Corr_gs = []
K_Corr_rs = []
A_gs = []
A_rs = []

for galaxy_name in all_results["Galaxy Name"]:
    tab = master_table[master_table["NAME"] == galaxy_name].iloc[0]
    DLs.append(tab["DL"])
    DAs.append(tab["DA"])
    zs.append(tab["REDSHIFT"])
    scales.append(tab["SCALE"])
    K_Corr_gs.append(tab["KCOR_G"])
    K_Corr_rs.append(tab["KCOR_R"])
    A_gs.append(tab["A_G"])
    A_rs.append(tab["A_R"])
    # if len(tab) == 0:
    #     print(tab)

all_results["DL"] = np.array(DLs) * u.Mpc
all_results["DA"] = np.array(DAs) * u.Mpc
all_results["z"] = np.array(zs, dtype=np.float64)
all_results["Scale"] = np.array(scales) * u.kpc/u.arcsec

A_gs = np.array(A_gs) 
A_rs = np.array(A_rs) 
K_Corr_gs = np.array(K_Corr_gs) 
K_Corr_rs = np.array(K_Corr_rs) 
tablen = len(all_results)
all_results["KCOR"] = np.zeros(tablen) 
all_results["A"] = np.zeros(tablen) 

band_m = all_results["band"] == "g"
all_results["KCOR"][band_m] = K_Corr_gs[band_m]
all_results["A"][band_m] = A_gs[band_m]
band_m = all_results["band"] == "r"
all_results["KCOR"][band_m] = K_Corr_rs[band_m]
all_results["A"][band_m] = A_rs[band_m]

all_results["App Mag"] = flux2ABmag(all_results["Flux"]*u.nmgy).value
all_results["Abs Mag"] = all_results["App Mag"] - astropy.coordinates.Distance(all_results["DL"]).distmod.value

all_results["Abs Mag Corr"] = all_results["Abs Mag"] - A_gs - K_Corr_gs
all_results = all_results[np.isfinite(all_results["DL"])]
all_results = all_results.group_by("Galaxy Name")

In [ ]:
SGAtable.columns
m = (SGAtable["Z_LEDA"] > 0) & (SGAtable["R_MAG_SB26"] != -1)
m2 = (SGAtable["Z_LEDA"] > 0) & (SGAtable["G_MAG_SB26"] != -1)


SGAtable["R_ABSMAG_SB26"] = -1*np.ones(len(SGAtable))
SGAtable["G_ABSMAG_SB26"] = -1*np.ones(len(SGAtable))

SGAtable["R_ABSMAG_SB26"][m] = (SGAtable["R_MAG_SB26"][m] *u.mag - cosmo.distmod(SGAtable["Z_LEDA"][m])).value
SGAtable["G_ABSMAG_SB26"][m2] = (SGAtable["G_MAG_SB26"][m2] *u.mag - cosmo.distmod(SGAtable["Z_LEDA"][m2])).value

In [ ]:
quantity_support()
fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=dpi, sharey='all')
ep=0.05

ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (-23, -12)
nbins = 25
bins = np.linspace(r[0], r[1], nbins) * u.ABmag

sgaval, bins = np.histogram(SGAtable["R_ABSMAG_SB26"] * u.ABmag, bins=bins)
for i in range(2):
    ax[i].stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_pol = np.zeros(np.size(bins)-1)

for k,gal_type in enumerate(gal_types):
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    polar_absmag = polar["Abs Mag"]
    host_absmag = host["Abs Mag"]


    polval, bins = np.histogram(polar_absmag, bins=bins)
    ax[0].stairs(polval/ngals+baseline_pol, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    # baseline_pol += polval/ngals

    hostval, bins = np.histogram(host_absmag, bins=bins)
    ax[1].stairs(hostval/ngals+baseline_host, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=False, alpha=1)
    # baseline_host += hostval/ngals


ax[0].set_xlabel(r"$M_r$ [mag]")
ax[1].set_xlabel(r"$M_r$ [mag]")
ax[0].set_ylabel("Fraction of Sample")
plt.suptitle(r"", y=0.93)
ax[0].legend()
ax[0].xaxis.set_inverted(True)
ax[1].xaxis.set_inverted(True)
plt.tight_layout()
plt.show()

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (-23, -12)
step = 0.6
bins = np.arange(r[0], r[1]+step, step) * u.ABmag

sgaval, bins = np.histogram(SGAtable["R_ABSMAG_SB26"] * u.ABmag, bins=bins)
plt.stairs(sgaval/len(SGAtable), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

polar_absmags = []

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for gal_type in gal_types:
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # z = master_table["Z"]
    polar_absmag = polar["Abs Mag"]
    polar_absmags.append(polar_absmag)
    host_absmag = host["Abs Mag"]


    polval, bins = np.histogram(polar_absmag, bins=bins)
    plt.stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Polar)", linewidth=linewidth, fill=True, alpha=0.6, hatch=".")
    baseline_polar += polval/ngals

    hostval, bins = np.histogram(host_absmag, bins=bins)
    plt.stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=True, alpha=0.6, hatch="X")
    baseline_host += hostval/ngals

# plt.hist(polar_absmags, stacked=True, label=gal_types, color=[type_colors[gal_type] for gal_type in gal_types])
plt.xlabel(r"$M_r$ [mag]")
plt.ylabel("Fraction of Sample")
plt.legend()
plt.show()

### 26th isophote radius

In [ ]:
import scipy

In [ ]:
def radius_of_isophote(mu_iso, mu_eff, n, r_eff):
    b_n = scipy.special.gammaincinv(2*n, 0.5)
    I_iso = nmgymagSB_to_fluxSB(mu_iso)
    I_eff = nmgymagSB_to_fluxSB(mu_eff)
    return r_eff * ((b_n-np.log(I_iso/I_eff))/b_n)**n

In [ ]:

# Example: radius of the 26 mag/arcsec^2 isophote
mu_26 = 26* u.mag("AB/arcsec**2")
mu_e = 27* u.mag("AB/arcsec**2")  # example effective surface brightness
n = 0.2
r_e = 60
r_26 = radius_of_isophote(mu_26, mu_e, n, r_e)
print(f"r_26 (semi-major axis) = {r_26:.2f}")

# quantity_support()
# plt.imshow(model(xx,yy), extent=(x[0],x[-1], y[0], y[-1]))

In [ ]:
all_results["mu_eff"] = nmgyfluxSB_to_magSB(all_results["I_e"] * u.nmgy/pixscale**2)
all_results["R26"] = radius_of_isophote(26*u.ABmag, all_results["mu_eff"], all_results["n"], all_results["r_e"]) * u.pix
all_results["R26"] = all_results["R26"].to(u.arcsec, equivalencies=u.pixel_scale(1*u.pix/(0.262*u.arcsec)))
all_results["R26"] = all_results["R26"]*all_results["Scale"]

m = (SGAtable["Z_LEDA"] >0) & (SGAtable["SMA_SB26"] != -1)
sga_sma_SB26_real = 1/cosmo.arcsec_per_kpc_proper(SGAtable["Z_LEDA"][m])*SGAtable["SMA_SB26"][m]*u.arcsec
# print(all_results["R26","n", "mu_eff"][all_results["R26"] != all_results["R26"]])

In [ ]:
# print(all_results["Scale","z"])
# print(cosmo.arcsec_per_kpc_proper(all_results["z"][np.isfinite(all_results["z"])]))
# print(1/cosmo.arcsec_per_kpc_proper(all_results["z"][all_results["z"]>0]))

In [ ]:
quantity_support()
# fig = plt.Figure(dpi=dpi)
fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=dpi, sharey='all')

# ngals = np.size(np.unique(all_results["Galaxy Name"]))
ep=0.3*u.kpc

r = (0, 100)
# step = 10
nbins = 30
bins = np.linspace(r[0], r[1], nbins) * u.kpc

sgaval, bins = np.histogram(sga_sma_SB26_real/2, bins=bins)
# plt.stairs(sgaval/len(SGAtable), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

for i in range(2):
    ax[i].stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for k,gal_type in enumerate(gal_types):
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # z = master_table["Z"]
    polar_r26 = polar["R26"]
    host_r26 = host["R26"]


    polval, bins = np.histogram(polar_r26, bins=bins)
    ax[0].stairs(polval/ngals+baseline_polar, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    # baseline_polar += polval/ngals

    hostval, bins = np.histogram(host_r26, bins=bins)
    ax[1].stairs(hostval/ngals+baseline_host, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=False, alpha=1)
    # baseline_host += hostval/ngals

ax[0].set_xlabel(r"$R26$ [kpc]")
ax[1].set_xlabel(r"$R26$ [kpc]")
ax[0].set_ylabel("Fraction of Sample")
ax[0].legend()
ax[0].set_xlim([0, 60])
ax[1].set_xlim([0, 60])
plt.tight_layout()
plt.show()

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (0, 100)
nbins = 30
bins = np.linspace(r[0], r[1], nbins) * u.kpc

sgaval, bins = np.histogram(sga_sma_SB26_real/2, bins=bins)
plt.stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for gal_type in gal_types:
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # z = master_table["Z"]
    polar_r26 = polar["R26"]
    host_r26 = host["R26"]


    polval, bins = np.histogram(polar_r26, bins=bins)
    plt.stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Polar)", linewidth=linewidth, fill=True, alpha=0.6, hatch=".")
    baseline_polar += polval/ngals

    hostval, bins = np.histogram(host_r26, bins=bins)
    plt.stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=True, alpha=0.6, hatch="X")
    baseline_host += hostval/ngals

plt.xlabel(r"$R26$ [kpc]")
plt.ylabel("Fraction of Sample")
plt.legend()
plt.show()

### Luminosity Ratio of Components (Though, I guess I don't have this for SGA galaxies?)

In [ ]:
def SersicFlux(I_e, r_e, n, ell):
    b_n = scipy.special.gammaincinv(2*n, 0.5)
    q = 1-ell
    L = 2*np.pi*n*np.exp(b_n)*I_e*(r_e**2)
    return q*L*scipy.special.gamma(2*n)/(b_n**(2*n))

In [ ]:
all_results["Flux2"] = SersicFlux(all_results["I_e"]*u.nmgy/pixscale**2, (all_results["r_e"]*u.pix).to(u.arcsec, equivalencies=u.pixel_scale(1*u.pix/(0.262*u.arcsec))), all_results["n"], all_results["ell"])
# all_results["Flux"] = all_results["Flux"] * u.nmgy

In [ ]:
plt.hist(SGAtable["R_ABSMAG_SB26"][SGAtable["R_ABSMAG_SB26"]!=-1])

### R26 Check with Photutils
Actually photutils seems to produce a generally worse result, but I think my alg works

In [ ]:
model = astropy.modeling.functional_models.Sersic2D(nmgymagSB_to_fluxSB(mu_e).value, r_eff=r_e, n=0.2, ellip=0.7)

sz = 100
x = np.linspace(-sz,sz,sz)
y = np.linspace(-sz,sz,sz)
xx,yy = np.meshgrid(x,y) 

In [ ]:
from photutils.isophote import EllipseGeometry
geometry = EllipseGeometry(x0=sz/2, y0=sz/2, sma=20, eps=0.5,
                           pa=np.deg2rad(0))

In [ ]:
from photutils.aperture import EllipticalAperture
aper = EllipticalAperture((geometry.x0, geometry.y0), geometry.sma,
                          geometry.sma * (1 - geometry.eps),
                          theta=geometry.pa)
fig, ax = plt.subplots()
ax.imshow(model(xx,yy), origin='lower')
aper.plot(color='white')

In [ ]:
for ellip in [0, 0.3, 0.5, 0.7]:
    model = astropy.modeling.functional_models.Sersic2D(nmgymagSB_to_fluxSB(mu_e).value, r_eff=r_e, n=0.2, ellip=ellip)
    model2 = astropy.modeling.functional_models.Sersic1D(nmgymagSB_to_fluxSB(mu_e).value, r_eff=r_e, n=0.2)

    sz = 100
    x = np.linspace(-sz,sz,sz)
    y = np.linspace(-sz,sz,sz)
    xx,yy = np.meshgrid(x,y) 

    plt.plot(x, nmgyfluxSB_to_magSB(model(xx,yy)[int(sz/2),:] * u.nmgy/u.arcsec**2), label=ellip)
    # plt.plot(model(xx,yy)[:,int(sz/2)], label=ellip)
    plt.plot(x, nmgyfluxSB_to_magSB(model2(x) * u.nmgy/u.arcsec**2), label="Model")
plt.xlim([0,100])
# plt.axvline(r_26_2)
plt.grid()
plt.legend()
plt.show()

In [ ]:
from photutils.isophote import Ellipse
ellipse = Ellipse(model(xx,yy), geometry=geometry)
try:
    isolist = ellipse.fit_image()
    t = isolist.to_table()
    t["mag"] = nmgyfluxSB_to_magSB(t["intens"] * u.nmgy/u.arcsec**2)
    # t["mag","sma"][:50]
    from photutils.isophote import build_ellipse_model
    model_image = build_ellipse_model(model(xx,yy).shape, isolist)
    plt.imshow(model_image)
    smas = np.linspace(10, 50, 5)
    for sma in smas:
        iso = isolist.get_closest(sma)
        x, y, = iso.sampled_coordinates()
        plt.plot(x, y, color='white')
except:
    isolist = None

### $g-r$ Color

In [ ]:
g_r_col = all_results["Abs Mag"][all_results["band"] == "g"]- all_results["Abs Mag"][all_results["band"] == "r"]
plt.hist(g_r_col)

In [ ]:
# res_fl = all_results.group_by(keys=["Galaxy Name", "band", "Function Label"])
# gal_col = all_results.group_by(keys=["Galaxy Name", "Function Label", "Galaxy Type"])
# gal_col = gal_col["Abs Mag", "Galaxy Name", "Function Label", "Galaxy Type"].groups.aggregate(np.subtract)
# gal_col["G-R"] = gal_col["Abs Mag"]
# print(all_results["Flux", "Galaxy Name", "band", "Function Label"][:200])
# print(all_results["Abs Mag", "Galaxy Name", "band", "Function Label"])

In [ ]:
g_band = all_results["band"] == "g"
r_band = all_results["band"] == "r"
gal_col = QTable(names=["Galaxy Name", "Function Label", "Galaxy Type", "G-R"], dtype=[np.str_, np.str_, np.str_, np.float128])

for galaxy_name in np.unique(all_results["Galaxy Name"]):
    for label in ["Host", "Polar"]:
        m = (all_results["Galaxy Name"] == galaxy_name) & (all_results["Function Label"] == label)

        g_r_col = all_results["Abs Mag Corr"][m&g_band] - all_results["Abs Mag Corr"][m&r_band]
        galaxy_type = all_results["Galaxy Type"][m][0]
        gal_col.add_row([galaxy_name, label, galaxy_type, np.nanmean(np.array(g_r_col))])

In [ ]:
# print(SGAtable[SGAtable["GALAXY"] == "PGC399039"]["SGA_ID"])
# TRACTOR[TRACTOR["SGA_ID"] == 612705]["EBV"]
# TRACTOR[TRACTOR["SGA_ID"] == 612705]["GAIA_A_G_VAL"]
# -2.5*np.log10(TRACTOR[TRACTOR["SGA_ID"] == 612704]["MW_TRANSMISSION_G"])

In [ ]:
sga_appmagcorr_g = SGAtable["G_MAG_SB26"] + 2.5*np.log10(TRACTOR["MW_TRANSMISSION_G"])
sga_appmagcorr_r = SGAtable["R_MAG_SB26"] + 2.5*np.log10(TRACTOR["MW_TRANSMISSION_R"])
sga_col = sga_appmagcorr_g - sga_appmagcorr_r

In [ ]:
quantity_support()

fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=dpi, sharey='all')

r = (-2, 3)
step = 0.1
nbins = 35 
bins = np.linspace(r[0], r[1], nbins) * u.ABmag

ep = 0.01

sgaval, bins = np.histogram(sga_col, bins=bins)
for i in range(2):
    ax[i].stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for k,gal_type in enumerate(gal_types):
    # polar_col = gal_col[(gal_col["Function Label"] == "Polar") & (gal_col["Galaxy Type"] == gal_type)]
    polar_col = gal_col[(gal_col["Function Label"] == "Polar") & (gal_col["Galaxy Type"] == gal_type)]
    host_col = gal_col[(gal_col["Function Label"] == "Host") & (gal_col["Galaxy Type"] == gal_type)]


    polval, bins = np.histogram(polar_col["G-R"], bins=bins)
    ax[0].stairs(polval/ngals+baseline_polar, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    # baseline_polar += polval/ngals

    # hostval, bins = np.histogram(host_r26, bins=bins)
    hostval, bins = np.histogram(host_col["G-R"], bins=bins)
    ax[1].stairs(hostval/ngals+baseline_host, edges=bins+ep*k, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=False, alpha=1)
    # baseline_host += hostval/ngals

ax[0].set_xlabel(r"$g-r$")
ax[1].set_xlabel(r"$g-r$")
ax[0].set_ylabel("Fraction of Sample")
ax[0].set_xlim([-0.5, 2])
ax[1].set_xlim([-0.5, 2])
ax[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (-2, 3)
step = 0.1
bins = np.arange(r[0], r[1]+step, step) * u.ABmag

sgaval, bins = np.histogram(sga_col, bins=bins)
plt.stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for gal_type in gal_types:
    # polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # z = master_table["Z"]
    polar_col = gal_col[(gal_col["Function Label"] == "Polar") & (gal_col["Galaxy Type"] == gal_type)]
    host_col = gal_col[(gal_col["Function Label"] == "Host") & (gal_col["Galaxy Type"] == gal_type)]


    polval, bins = np.histogram(polar_col["G-R"], bins=bins)
    plt.stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Polar)", linewidth=linewidth, fill=True, alpha=0.6, hatch=".")
    baseline_polar += polval/ngals

    # hostval, bins = np.histogram(host_r26, bins=bins)
    hostval, bins = np.histogram(host_col["G-R"], bins=bins)
    plt.stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=True, alpha=0.6, hatch="X")
    baseline_host += hostval/ngals

plt.xlabel(r"$g-r$ [mag]")
plt.ylabel("Fraction of Sample")
plt.xlim([-0.5, 2])
plt.legend()
plt.show()

Make seperate plots: PS and Host plots seperate

2 plots: SGA galaxies, each PS type properties (each Sersic parameter); Same but with just Host structure and SGA

### Sersic Index (in $r$ band)

In [ ]:
quantity_support()
fig, ax = plt.subplots(1,2, figsize=(16,8), dpi=dpi, sharey='all')
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (0, 11)
nbins = 30
bins = np.linspace(r[0], r[1], nbins)
ep=0.05

sgaval, bins = np.histogram(TRACTOR["SERSIC"], bins=bins)
for i in range(2):
    ax[i].stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for k,gal_type in enumerate(gal_types):
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]


    polval, bins = np.histogram(polar["n"], bins=bins)
    # ax[0].stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    ax[0].stairs(polval/ngals+baseline_polar,edges=bins+k*ep, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type}", linewidth=linewidth, fill=False, alpha=1)
    # baseline_polar += polval/ngals

    hostval, bins = np.histogram(host["n"], bins=bins)
    # ax[1].stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=False, alpha=1)
    ax[1].stairs(hostval/ngals+baseline_host, edges=bins+k*ep, color=type_colors[gal_type], edgecolor=type_colors[gal_type], ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=False, alpha=1)
    # baseline_host += hostval/ngals

ax[0].set_xlabel(r"Sersic Index")
ax[1].set_xlabel(r"Sersic Index")
ax[0].set_ylabel("Fraction of Sample")
ax[0].legend()
ax[0].set_xlim([0,7.5])
ax[1].set_xlim([0,7.5])
plt.tight_layout()
plt.show()

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (0, 11)
step = 0.5
bins = np.arange(r[0], r[1]+step, step)

sgaval, bins = np.histogram(TRACTOR["SERSIC"], bins=bins)
plt.stairs(sgaval/np.sum(sgaval), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for gal_type in gal_types:
    polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]


    polval, bins = np.histogram(polar["n"], bins=bins)
    plt.stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Polar)", linewidth=linewidth, fill=True, alpha=0.6, hatch=".")
    baseline_polar += polval/ngals

    hostval, bins = np.histogram(host["n"], bins=bins)
    plt.stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=True, alpha=0.6, hatch="X")
    baseline_host += hostval/ngals

plt.xlabel(r"n")
plt.ylabel("Fraction of Sample")
plt.legend()
plt.show()

### Color-Color for Host and Polar Structure

In [ ]:
type_marker = {
    "PR": "o",
    "PH": "D",
    "PB": "^"
}

In [ ]:
quantity_support()
fig = plt.Figure(dpi=dpi)
ngals = np.size(np.unique(all_results["Galaxy Name"]))

r = (-2, 3)
step = 0.3
bins = np.arange(r[0], r[1]+step, step) * u.ABmag

# sgaval, bins = np.histogram(sga_col, bins=bins)
# plt.stairs(sgaval/len(SGAtable), edges=bins, color="grey", edgecolor="orange", ls="-", label=f"SGA", linewidth=linewidth, fill=True, alpha=1)

baseline_host = np.zeros(np.size(bins)-1)
baseline_polar = np.zeros(np.size(bins)-1)
for gal_type in gal_types:
    # polar = all_results[(all_results["Function Label"] == "Polar") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # host = all_results[(all_results["Function Label"] == "Host") & (all_results["Galaxy Type"]==gal_type) & (all_results["band"] == "r")]
    # z = master_table["Z"]
    polar_col = gal_col[(gal_col["Function Label"] == "Polar") & (gal_col["Galaxy Type"] == gal_type)]
    host_col = gal_col[(gal_col["Function Label"] == "Host") & (gal_col["Galaxy Type"] == gal_type)]


    # polval, bins = np.histogram(polar_col["G-R"], bins=bins)
    # plt.stairs(polval/ngals+baseline_polar, baseline=baseline_polar, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Polar)", linewidth=linewidth, fill=True, alpha=0.6, hatch=".")
    # baseline_polar += polval/ngals

    # # hostval, bins = np.histogram(host_r26, bins=bins)
    # hostval, bins = np.histogram(host_col["G-R"], bins=bins)
    # plt.stairs(hostval/ngals+baseline_host,baseline=baseline_host, edges=bins, color=type_colors[gal_type], edgecolor="black", ls=type_ls[gal_type], label=f"{gal_type} (Host)", linewidth=linewidth, fill=True, alpha=0.6, hatch="X")
    # baseline_host += hostval/ngals

    plt.scatter(host_col["G-R"], polar_col["G-R"], label=gal_type, color=type_colors[gal_type], marker=type_marker[gal_type])

plt.xlabel(r"$g-r$ of Host")
plt.ylabel(r"$g-r$ of PS")
plt.xlim([0,2])
plt.ylim([0,2])
plt.legend()
plt.show()

### Color Magnitude

In [ ]:
band_group = all_results.group_by(keys=["Galaxy Name", "Galaxy Type", "band", "A", "KCOR", "z", "DL"])
band_group = band_group["Galaxy Name", "Galaxy Type", "band", "Flux", "A", "KCOR", "z", "DL"].groups.aggregate(np.sum)

band_group["App Mag"] = flux2ABmag(band_group["Flux"] * u.nmgy).value
band_group["Abs Mag"] = band_group["App Mag"] - astropy.coordinates.Distance(band_group["DL"]).distmod.value
band_group["Abs Mag Corr"] = band_group["Abs Mag"] - band_group["A"] - band_group["KCOR"]

band_group_col = band_group.group_by(keys=["Galaxy Name", "Galaxy Type"])
band_group_col = band_group_col["Galaxy Name", "Galaxy Type", "Abs Mag Corr"].groups.aggregate(np.subtract)
band_group_col["G-R"] = band_group_col["Abs Mag Corr"]

In [ ]:
import matplotlib

In [ ]:
# fig = plt.figure(figsize=(8,8), dpi=100)
fig, ax = plt.subplots(1,3, figsize=(24,8), dpi=dpi, sharey='all', layout="constrained")

m = (SGAtable["Z_LEDA"] > 0) & (SGAtable["R_MAG_SB26"] != -1)
m2 = (SGAtable["Z_LEDA"] > 0) & (SGAtable["G_MAG_SB26"] != -1)
m3 = (SGAtable["R_ABSMAG_SB26"] < -16) & (SGAtable["R_ABSMAG_SB26"] > -25) & (sga_col > 0.1) & (sga_col < 1.3)

# plt.hexbin(SGAtable["R_ABSMAG_SB26"][m & m3], sga_col[m & m3], gridsize=100, cmap="gray_r", norm=matplotlib.colors.LogNorm())

for i in range(3):
    ax[i].set_xlabel(r"$M_r$ [mag]")
    ax[i].xaxis.set_inverted(True)
    ax[i].set_xlim([-16, -24])
    ax[i].set_ylim([0.1, 1.3])
    hexbin = ax[i].hexbin(SGAtable["R_ABSMAG_SB26"][m & m3], sga_col[m & m3], gridsize=100, cmap="magma")#, norm=matplotlib.colors.LogNorm())

for gal_type in gal_types:
    m4 = (band_group["band"] == "r") 
    m5 = (band_group["Galaxy Type"] == gal_type)
    ax[0].scatter(band_group["Abs Mag"][m4 & m5], band_group_col["G-R"][band_group_col["Galaxy Type"] == gal_type], color=type_colors[gal_type], label=gal_type, marker=type_marker[gal_type])

    ax[1].scatter(all_results["Abs Mag Corr"][(all_results["band"] == "r") & (all_results["Galaxy Type"] == gal_type) & (all_results["Function Label"] == "Polar")], gal_col["G-R"][(gal_col["Galaxy Type"] == gal_type) & (gal_col["Function Label"] == "Polar")],
                  color=type_colors[gal_type], label=gal_type, marker=type_marker[gal_type])

    ax[2].scatter(all_results["Abs Mag Corr"][(all_results["band"] == "r") & (all_results["Galaxy Type"] == gal_type) & (all_results["Function Label"] == "Host")], gal_col["G-R"][(gal_col["Galaxy Type"] == gal_type) & (gal_col["Function Label"] == "Host")],
                  color=type_colors[gal_type], label=gal_type, marker=type_marker[gal_type])

# plt.scatter(all_results["Abs Mag Corr"])
# plt.xlim([-15, -25])
# plt.ylim([0.2, 1.2])
# ax = plt.gca()

# cb_ax = fig.add_axes([.91,.124,.04,.754])
cb_ax = fig.add_axes([1,0.06,.02,1-0.06])
fig.colorbar(hexbin,orientation='vertical',cax=cb_ax, label="Number of Galaxies (SGA)")
# plt.colorbar(label="Number of Galaxies (SGA)", mappable=hexbin)

ax[0].legend()
ax[0].set_ylabel(r"$g-r$")
# plt.tight_layout()
plt.show()